In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from IPython.display import Image, display
from dotenv import load_dotenv
import os
import requests

load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_REPO  = "rahul8879/e-comm-agentic-demo"
PR_NUMBER    = 1

HEADERS = {"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"}
BASE_URL = f"https://api.github.com/repos/{GITHUB_REPO}"

r = requests.get(BASE_URL, headers=HEADERS)
print(f"Repo: {r.json().get('full_name')}")
print(f"Status: {r.status_code}")

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo: rahul8879/e-comm-agentic-demo
Status: 200


In [2]:
# pip install langgraph-checkpoint-postgres "psycopg[binary,pool]"

In [3]:
from langgraph.store.postgres import PostgresStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
DB_URI = os.getenv("NEON_DB_URI")

_store_cm = PostgresStore.from_conn_string(
    DB_URI,
    index ={
        "dims": 1536,
        "embed":embeddings
    }
)

store = _store_cm.__enter__()

store.setup()


In [4]:
def semantic_ns() -> tuple:
    return ("codesentinel", GITHUB_REPO, "semantic")

def procedural_ns() -> tuple:
    return ("codesentinel", GITHUB_REPO, "procedural")

def episodic_ns(author: str) -> tuple:
    return ("codesentinel", GITHUB_REPO, "episodic", author)


SEMANTIC   = semantic_ns()
PROCEDURAL = procedural_ns()

In [5]:
import uuid
rules = [
    "This repo requires parameterized SQL queries. Never build SQL with f-strings or % formatting.",
    "Passwords must be hashed with bcrypt or Argon2. MD5/SHA1 are banned.",
    "Secrets (keys, passwords) must come from environment variables, never hardcoded.",
]

for i in rules:
    store.put(SEMANTIC,str(uuid.uuid4()),{"standard": i})

In [6]:
from pathlib import Path

In [7]:
SKILLS_DIR = Path("skills")
for skill_path in sorted(SKILLS_DIR.glob("*.md")):
    store.put(PROCEDURAL, skill_path.stem, {
        "skill_name": skill_path.name,
        "content": skill_path.read_text(),
    })

In [ ]:
# SEMANTIC

('codesentinel', 'rahul8879/e-comm-agentic-demo', 'semantic')

In [ ]:
("codesentinel", GITHUB_REPO, "semantic")

In [ ]:
# prefix = codesentinel-e_commerce_demo_semantic
# key = 1
# value ={"standard ": "Repo code should follow xyz pattern"}
# prefix = codesentinel-e_commerce_demo_semantic
# key = 2
# value ={"standard ": "SQL code should not have any hard coded value"}

In [ ]:
# ("codesentinel", GITHUB_REPO, "episodic", author)
# prefix = codesentinel-e_commerce_demo_episodic_rahul8879
# key = 1
# value ={"what ": "Repo code should follow xyz pattern"
#         "when ": "2023-06-01"}